# Part C: LangChain + RAG - M&A Knowledge Assistant

Build a Q&A system that answers questions **only** from the M&A Playbook PDF. No internet search, no external knowledge - if the answer isn't in the retrieved context, the assistant must say so.

In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import warnings
warnings.filterwarnings('ignore')

print('Libraries imported successfully!')

## TASK 12: Load & Process the M&A Playbook PDF

In [ ]:
PDF_PATH = "../M&A Playbook_ Comprehensive Guide to Deals and Integration.pdf"

loader = PyPDFLoader(PDF_PATH)
documents = loader.load()

full_text = ""
for page in documents:
    full_text += page.page_content + "\n"

print(f"Total pages: {len(documents)}")
print(f"Total characters: {len(full_text)}")
print(full_text[:1000])

### Chunking experimentation

We try a few chunk sizes to see how they affect the number of chunks and the amount of context passed to the LLM per retrieval, then pick one to move forward with.

In [ ]:
for size, overlap in [(500, 50), (1000, 150), (2000, 200)]:
    splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=overlap)
    test_chunks = splitter.split_documents(documents)
    avg_len = sum(len(c.page_content) for c in test_chunks) / len(test_chunks)
    print(f"chunk_size={size:5d}  overlap={overlap:4d}  -> {len(test_chunks):3d} chunks, avg chunk length={avg_len:.0f}")

**Decision: chunk_size=1000, chunk_overlap=150**

- 500 chars produces many small chunks that can split a single idea (e.g. a step in a deal process) across chunks, losing context.
- 2000 chars keeps ideas together but retrieves large blocks of mostly-irrelevant text per query, diluting the context sent to the LLM.
- 1000 chars with a 150-char overlap keeps each chunk focused on one topic/section while the overlap preserves continuity across chunk boundaries (e.g. a sentence that starts near the end of one chunk).

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)

chunks = text_splitter.split_documents(documents)
print(f"Final number of chunks: {len(chunks)}")

## TASK 13: Generate Embeddings

In [ ]:
embedding = GoogleGenerativeAIEmbeddings(model="models/embedding-001")

sample_vector = embedding.embed_query(chunks[0].page_content)
print(f"Embedding dimension: {len(sample_vector)}")

## TASK 14: Vector Store (FAISS)

In [ ]:
vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embedding
)

print("FAISS vector store built with", vector_store.index.ntotal, "vectors")

In [ ]:
query = "What are the key steps in the due diligence process?"

results = vector_store.similarity_search_with_score(query, k=5)

for i, (doc, score) in enumerate(results, start=1):
    print(f"result {i}")
    print(f"similarity_score: {score}")
    print(f"page: {doc.metadata.get('page')}")
    print(f"content: {doc.page_content[:300]}\n")

## TASK 15: Build the RAG Pipeline

In [ ]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

LLM = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

## TASK 16: Prompt Engineering

The prompt instructs the model to rely only on the retrieved context, avoid hallucination, and explicitly say when information is unavailable. It also requires the response to cite supporting evidence and source pages.

In [ ]:
rag_prompt = ChatPromptTemplate.from_template(
    """
    You are an M&A Knowledge Assistant. Answer the user's question using ONLY the
    retrieved context below, which comes exclusively from the M&A Playbook.

    Rules:
    - Do not use any outside knowledge or make assumptions beyond the context.
    - If the answer is not present in the context, respond exactly with:
      "I couldn't find this information in the M&A Playbook."
    - Never fabricate facts, numbers, or page references.

    Retrieved context:
    {context}

    Question: {question}

    Provide your response in this format:
    1. Answer
    2. Supporting Evidence (quote from the context)
    3. Source Page(s)
    """
)

rag_chain = rag_prompt | LLM | StrOutputParser()

In [ ]:
def ask(question):
    retrieved_docs = retriever.invoke(question)
    context = "\n\n".join(
        f"[Page {doc.metadata.get('page')}] {doc.page_content}" for doc in retrieved_docs
    )
    response = rag_chain.invoke({"context": context, "question": question})
    print(response)
    return response

_ = ask("What are the key steps in the due diligence process?")

In [ ]:
_ = ask("What is the capital of France?")

The second question is unrelated to the M&A Playbook, so the assistant should decline to answer rather than hallucinate a response - this confirms the hallucination guardrail is working.

## Questions

**Why chunk documents?**
LLMs and embedding models have limited context windows, and semantic search works best over focused, topically-coherent text spans. Chunking breaks a large document into pieces small enough to embed and retrieve individually, so only the relevant portion of the document is passed to the LLM instead of the entire PDF.

**Effects of too-small/too-large chunks?**
- Too small: ideas get split mid-thought, embeddings capture incomplete context, and retrieval may miss the full answer even if a fragment matches the query.
- Too large: each chunk mixes multiple topics, diluting the embedding's specificity and returning bloated context that adds noise and increases token cost/latency for the LLM.

**How do embeddings enable semantic search?**
Embeddings map text into a numeric vector space where semantically similar text ends up close together, regardless of exact wording. This lets retrieval match a query to relevant chunks based on meaning (e.g. "due diligence steps" matching a chunk about "deal evaluation process") rather than requiring exact keyword overlap, which is what FAISS's similarity search exploits.